# Binary Oil-Spill Classification from Sentinel-1 SAR

A complete binary image-classification workflow for the CSIRO dataset. It predicts only no_oil or oil; it does not create masks, boundaries, area, coordinates, vessel identity, or AIS correlations. The dataset is inspected before use because labels may be directories or files named 0 and 1 containing relative paths.


## 1. Imports and configuration


In [ ]:
from pathlib import Path
import os, random, zipfile, warnings, pickle, shutil, time
from collections import Counter
import numpy as np
from PIL import Image, UnidentifiedImageError
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, ConfusionMatrixDisplay

SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DATASET_FILENAME='2022-12-15_Blondeau-Patissier_David_57430v1.zip'
# Dataset location supplied by the user.
# Directory Structure - oil_spill_work - contains - code.ipynb file and .zip S1 SAR dataset
DATASET_PATH=Path(r'E:\oil_spill_work\2022-12-15_Blondeau-Patissier_David_57430v1.zip')
if not DATASET_PATH.is_file():
    raise FileNotFoundError(f'Dataset path does not exist: {DATASET_PATH.resolve()}')
print('Using dataset:',DATASET_PATH.resolve())
WORK_DIR=Path('./oil_spill_work_cpu'); IMAGE_SIZE=224; BATCH_SIZE=32; LEARNING_RATE=1e-3
EPOCHS=15; PATIENCE=4; MODEL_NAME='efficientnet_b0'; NUM_WORKERS=2
CPU_THREADS = min(8, os.cpu_count() or 1)
torch.set_num_threads(CPU_THREADS)
CHECKPOINT_PATH=WORK_DIR/'best_model_cpu.pth'; PICKLE_PATH=WORK_DIR/'best_model_cpu.pkl'; DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'CPU threads: {CPU_THREADS}; DataLoader workers: {NUM_WORKERS}')
CLASS_NAMES=['no_oil','oil']; print('Device:',DEVICE)

if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))






## 2. Dataset path configuration and structure inspection


In [ ]:
def prepare_root(path, work_dir):
    path=Path(path).expanduser()
    if not path.exists(): raise FileNotFoundError(f'Dataset path does not exist: {path.resolve()}')
    if not (path.is_file() and path.suffix.lower()=='.zip'): raise ValueError('DATASET_PATH must be the uploaded outer ZIP file.')
    out=work_dir/'extracted_dataset'; out.mkdir(parents=True,exist_ok=True); marker=out/'.outer_extracted'
    if not marker.exists():
        with zipfile.ZipFile(path) as zf: zf.extractall(out)
        marker.touch()
    return out

# Exact CSIRO archive layout:
# outer ZIP/data/S1SAR_UnBalanced_400by400_Class_0.zip/0 = no_oil
# outer ZIP/data/S1SAR_UnBalanced_400by400_Class_1.zip/1 = oil
DATASET_ROOT=prepare_root(DATASET_PATH,WORK_DIR)
data_dir=DATASET_ROOT/'data'
CLASS_ARCHIVES={0:data_dir/'S1SAR_UnBalanced_400by400_Class_0.zip',1:data_dir/'S1SAR_UnBalanced_400by400_Class_1.zip'}
CLASS_DIRS={}
for label,archive in CLASS_ARCHIVES.items():
    if not archive.exists(): raise FileNotFoundError(f'Missing class archive: {archive}')
    target=DATASET_ROOT/f'__class_{label}'; marker=target/'.extracted'; expected=target/str(label)
    # Rebuild an incomplete/stale extraction from a previous notebook run.
    if marker.exists() and not expected.is_dir():
        shutil.rmtree(target)
    target.mkdir(parents=True,exist_ok=True)
    if not marker.exists():
        with zipfile.ZipFile(archive) as zf: zf.extractall(target)
        marker.touch()
    if not expected.is_dir(): raise RuntimeError(f'Expected class folder {label} inside {archive.name}')
    CLASS_DIRS[label]=expected
print('Dataset root:',DATASET_ROOT.resolve())
print('Class 0 directory (no_oil):',CLASS_DIRS[0])
print('Class 1 directory (oil):',CLASS_DIRS[1])

# Class entries are now explicit directories, not guessed from the filesystem.
ENTRIES={'0':CLASS_DIRS[0],'1':CLASS_DIRS[1]}
print('Class entries:',ENTRIES)



## 3. Dataset validation


In [ ]:
EXTS={'.jpg','.jpeg','.png','.tif','.tiff','.bmp','.webp'}
def resolve_ref(ref,list_file,root):
    ref=ref.strip().strip(chr(34)+chr(39)); candidates=[list_file.parent/ref,root/ref]
    for p in candidates:
        if p.exists(): return p.resolve()
    return candidates[1].resolve()
def collect(entry,label,root):
    samples=[]; missing=[]; ignored=[]
    if entry.is_dir():
        for p in entry.rglob('*'):
            if p.is_file() and p.suffix.lower() in EXTS: samples.append((p.resolve(),label))
            elif p.is_file(): ignored.append(p)
    elif entry.is_file():
        for n,line in enumerate(entry.read_text(errors='replace').splitlines(),1):
            if not line.strip() or line.lstrip().startswith('#'): continue
            p=resolve_ref(line,entry,root)
            if not p.exists(): missing.append((n,line))
            elif p.suffix.lower() in EXTS: samples.append((p,label))
            else: ignored.append(p)
    else:
        raise RuntimeError(f'Expected class directory or path-list file, but found: {entry}')
    return samples,missing,ignored
all_samples=[]; missing=[]; ignored=[]
for label in (0,1):
    s,m,i=collect(ENTRIES[str(label)],label,DATASET_ROOT); all_samples+=s; missing += [(label,*x) for x in m]; ignored+=i
valid=[]; corrupt=[]; dims=Counter(); modes=Counter()
for p,y in all_samples:
    try:
        with Image.open(p) as im: im.verify()
        with Image.open(p) as im: dims[im.size]+=1; modes[im.mode]+=1
        valid.append((p,y))
    except (OSError,UnidentifiedImageError,ValueError) as e: corrupt.append((p,str(e)))
print('Samples by class:',Counter(CLASS_NAMES[y] for _,y in valid)); print('Missing:',len(missing),'Corrupt:',len(corrupt),'Ignored:',len(ignored)); print('Dimensions:',dims.most_common(10)); print('Modes:',modes)
for y in (0,1): print(CLASS_NAMES[y],'examples:',[str(p) for p,l in valid if l==y][:5])
if not valid or any(not any(y==i for _,y in valid) for i in (0,1)): raise RuntimeError('Readable images were not found in both classes.')



## 4. Sample images and stratified split


In [ ]:
fig,ax=plt.subplots(2,4,figsize=(13,6))
for y in (0,1):
    for j,(p,_) in enumerate([(p,l) for p,l in valid if l==y][:4]):
        with Image.open(p) as im: ax[y,j].imshow(im.convert('L'),cmap='gray')
        ax[y,j].set_title(CLASS_NAMES[y]); ax[y,j].axis('off')
plt.tight_layout(); plt.show()
paths=np.array([str(p) for p,y in valid]); labels=np.array([y for p,y in valid])
train_p,temp_p,train_y,temp_y=train_test_split(paths,labels,test_size=.30,stratify=labels,random_state=SEED)
val_p,test_p,val_y,test_y=train_test_split(temp_p,temp_y,test_size=.50,stratify=temp_y,random_state=SEED)
for name,ys in [('train',train_y),('validation',val_y),('test',test_y)]: print(name,len(ys),Counter(CLASS_NAMES[int(y)] for y in ys))


## 5. Preprocessing, datasets, and DataLoaders


In [ ]:
MEAN=[.485,.456,.406]; STD=[.229,.224,.225]
train_tf=transforms.Compose([transforms.Grayscale(3),transforms.Resize((IMAGE_SIZE,IMAGE_SIZE)),transforms.RandomHorizontalFlip(),transforms.RandomVerticalFlip(),transforms.RandomRotation(8),transforms.ToTensor(),transforms.Normalize(MEAN,STD)])
eval_tf=transforms.Compose([transforms.Grayscale(3),transforms.Resize((IMAGE_SIZE,IMAGE_SIZE)),transforms.ToTensor(),transforms.Normalize(MEAN,STD)])
class SARDataset(Dataset):
    def __init__(self,paths,labels,tf): self.paths=list(paths); self.labels=np.asarray(labels,dtype=np.int64); self.tf=tf
    def __len__(self): return len(self.paths)
    def __getitem__(self,i):
        with Image.open(self.paths[i]) as im: image=im.convert('L')
        return self.tf(image),int(self.labels[i])
train_ds=SARDataset(train_p,train_y,train_tf); val_ds=SARDataset(val_p,val_y,eval_tf); test_ds=SARDataset(test_p,test_y,eval_tf)
kw=dict(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=DEVICE.type == 'cuda'
)
train_loader=DataLoader(train_ds,shuffle=True,**kw); val_loader=DataLoader(val_ds,shuffle=False,**kw); test_loader=DataLoader(test_ds,shuffle=False,**kw)
print('Dataset sizes:',len(train_ds),len(val_ds),len(test_ds))



## 6. Model initialization and required smoke test


In [ ]:
def build_model():
    try: model=models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    except Exception as e: warnings.warn('Pretrained weights unavailable: '+str(e)); model=models.efficientnet_b0(weights=None)
    model.classifier[1]=nn.Linear(model.classifier[1].in_features,2)
    for parameter in model.features.parameters(): parameter.requires_grad=False
    return model.to(DEVICE)
model=build_model(); counts=np.bincount(train_y,minlength=2); weights=len(train_y)/(2*np.maximum(counts,1))
criterion=nn.CrossEntropyLoss(weight=torch.tensor(weights,dtype=torch.float32,device=DEVICE))
optimizer=torch.optim.AdamW(model.classifier.parameters(),lr=LEARNING_RATE)
total_params=sum(p.numel() for p in model.parameters()); trainable_params=sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters: {total_params:,}'); print(f'Trainable parameters: {trainable_params:,} (classifier only)')
smoke=SARDataset([next(p for p,y in valid if y==i) for i in (0,1)],[0,1],eval_tf); x=torch.stack([smoke[i][0] for i in range(2)]).to(DEVICE)
with torch.inference_mode(): out=model(x)
assert x.shape==(2,3,IMAGE_SIZE,IMAGE_SIZE) and out.shape==(2,2); print('Smoke test passed:',x.shape,'->',out.shape,'labels map to',CLASS_NAMES)




## CPU optimization notes

This version is configured for an older Intel i5 CPU: it forces CPU execution, uses a conservative zero-worker DataLoader to avoid multiprocessing overhead, limits PyTorch threads to at most four, disables pinned memory, freezes EfficientNet features, trains only the small classifier, times every epoch, and uses inference_mode for validation/testing. It keeps the same stratified split and held-out test methodology. The expected improvement is typically several times faster than full EfficientNet fine-tuning on CPU, although actual time depends on storage, RAM, and Python/PyTorch installation.


## 7. Training, validation, and checkpointing


In [ ]:
def epoch_run(model,loader,criterion,opt=None):
    training=opt is not None; model.train(training); loss_sum=correct=n=0
    for x,y in loader:
        x,y=x.to(DEVICE),y.to(DEVICE)
        if training: opt.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            logits=model(x); loss=criterion(logits,y)
            if training: loss.backward(); opt.step()
        loss_sum+=loss.item()*len(y); correct+=(logits.argmax(1)==y).sum().item(); n+=len(y)
    return loss_sum/n,correct/n

WORK_DIR.mkdir(parents=True,exist_ok=True)
# A valid pickle prevents unnecessary retraining on later notebook runs.
if PICKLE_PATH.exists():
    with open(PICKLE_PATH,'rb') as f: saved=pickle.load(f)
    model.load_state_dict(saved['model_state_dict']); history=saved['history']; best=saved['best_validation_loss']; best_epoch=saved['best_epoch']
    print('Loaded existing CPU model from:',PICKLE_PATH); print('Training skipped. Best epoch:',best_epoch)
else:
    history={k:[] for k in ['train_loss','train_accuracy','val_loss','val_accuracy']}; best=float('inf'); stale=0; best_epoch=0
    for ep in range(1,EPOCHS+1):
        started=time.perf_counter(); a,b=epoch_run(model,train_loader,criterion,optimizer); c,d=epoch_run(model,val_loader,criterion); elapsed=time.perf_counter()-started
        for k,v in zip(history,[a,b,c,d]): history[k].append(v)
        print(f'Epoch {ep:02d}: {elapsed/60:.2f} min | train loss {a:.4f} acc {b:.4f} | val loss {c:.4f} acc {d:.4f}')
        if c<best:
            best=c; best_epoch=ep; stale=0
            torch.save({'model_state_dict':model.state_dict(),'model_name':MODEL_NAME,'image_size':IMAGE_SIZE,'class_names':CLASS_NAMES,'val_loss':c,'epoch':ep},CHECKPOINT_PATH)
        else:
            stale+=1
            if stale>=PATIENCE: print('Early stopping'); break
    payload={'model_state_dict':{k:v.detach().cpu() for k,v in model.state_dict().items()},'history':history,'best_validation_loss':best,'best_epoch':best_epoch,'model_name':MODEL_NAME,'image_size':IMAGE_SIZE,'class_names':CLASS_NAMES,'trainable_parameters':trainable_params,'total_parameters':total_params}
    with open(PICKLE_PATH,'wb') as f: pickle.dump(payload,f,protocol=pickle.HIGHEST_PROTOCOL)
    print('Training complete. Persistent CPU pickle saved to:',PICKLE_PATH)
torch.save({'model_state_dict':model.state_dict(),'model_name':MODEL_NAME,'image_size':IMAGE_SIZE,'class_names':CLASS_NAMES,'val_loss':best,'epoch':best_epoch},CHECKPOINT_PATH)
print('Best checkpoint:',CHECKPOINT_PATH)


## 8. Held-out test evaluation and plots


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    ConfusionMatrixDisplay
)

# Load best trained model
with open(PICKLE_PATH, 'rb') as f:
    ckpt = pickle.load(f)

model.load_state_dict(ckpt['model_state_dict'])
model.eval()

truth = []
pred = []
oil_prob = []

# Test-set prediction
with torch.inference_mode():

    for x, y in test_loader:

        x = x.to(DEVICE)

        prob = torch.softmax(
            model(x),
            dim=1
        )

        truth += y.tolist()
        pred += prob.argmax(1).cpu().tolist()
        oil_prob += prob[:, 1].cpu().tolist()


# ------------------------------------------------------------
# Classification metrics
# ------------------------------------------------------------

cm = confusion_matrix(
    truth,
    pred,
    labels=[0, 1]
)

accuracy = accuracy_score(truth, pred)
precision = precision_score(
    truth,
    pred,
    zero_division=0
)
recall = recall_score(
    truth,
    pred,
    zero_division=0
)
f1 = f1_score(
    truth,
    pred,
    zero_division=0
)

roc_auc = roc_auc_score(
    truth,
    oil_prob
)

metrics = {
    'accuracy': accuracy,
    'precision_oil': precision,
    'recall_oil': recall,
    'f1_oil': f1,
    'roc_auc': roc_auc,
    'false_positives_no_oil_to_oil': int(cm[0, 1]),
    'false_negatives_oil_to_no_oil': int(cm[1, 0])
}

print('Test Metrics:')
for name, value in metrics.items():
    if isinstance(value, float):
        print(f'{name}: {value:.4f}')
    else:
        print(f'{name}: {value}')


# ------------------------------------------------------------
# ROC CURVE
# ------------------------------------------------------------

fpr, tpr, thresholds = roc_curve(
    truth,
    oil_prob
)

plt.figure(figsize=(7, 6))

plt.plot(
    fpr,
    tpr,
    label=f'ROC curve (AUC = {roc_auc:.4f})',
    linewidth=2
)

# Random classifier / no-skill line
plt.plot(
    [0, 1],
    [0, 1],
    linestyle='--',
    label='Random classifier'
)

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Oil Spill Classification')

plt.legend(loc='lower right')
plt.grid(alpha=0.3)

plt.show()


# ------------------------------------------------------------
# CONFUSION MATRIX
# ------------------------------------------------------------

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=CLASS_NAMES
).plot(
    cmap='Blues',
    values_format='d'
)

plt.title('Held-out Test Confusion Matrix')
plt.show()


# ------------------------------------------------------------
# TRAINING / VALIDATION LOSS AND ACCURACY
# ------------------------------------------------------------

fig, ax = plt.subplots(1, 2, figsize=(12, 4))

e = range(
    1,
    len(history['train_loss']) + 1
)

# Loss
ax[0].plot(
    e,
    history['train_loss'],
    label='Train'
)

ax[0].plot(
    e,
    history['val_loss'],
    label='Validation'
)

ax[0].set_title('Loss')
ax[0].set_xlabel('Epoch')
ax[0].set_ylabel('Loss')
ax[0].legend()


# Accuracy
ax[1].plot(
    e,
    history['train_accuracy'],
    label='Train'
)

ax[1].plot(
    e,
    history['val_accuracy'],
    label='Validation'
)

ax[1].set_title('Accuracy')
ax[1].set_xlabel('Epoch')
ax[1].set_ylabel('Accuracy')
ax[1].legend()

plt.tight_layout()
plt.show()